# B&H (2009) Strict Replication Analysis
This notebook implements the color decoding analysis pipeline as described in Brouwer & Heeger (2009, J. Neurosci.).
The only deviation is the use of atlas-based ROIs (V1-V4) instead of retinotopy.

**Analysis Steps:**
1. **Setup & Data Prep**: Load paths, define parameters, and prepare data loaders.
2. **First-Level GLM (FIR)**: Run a voxel-wise FIR GLM to estimate response amplitudes (β coefficients) for each color condition.
3. **ROI Masking**: Load atlas-based ROI masks (V1-V4) and extract β values from them.
4. **Forward Encoding Model**: Implement the leave-one-run-out cross-validation to train and test the forward encoding model.
5. **Significance Testing**: Perform a permutation test to assess the statistical significance of the decoding accuracy.
6. **PCA Visualization**: As a complementary analysis, run PCA on the response patterns to visualize the color space.

## 1. Setup & Data Prep
분석에 필요한 라이브러리를 임포트하고, 파일 경로, 파라미터 등 기본 환경을 설정합니다.

In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.glm.first_level import FirstLevelModel
from nilearn.image import clean_img, index_img
from nilearn.maskers import NiftiMasker
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# --- 기본 설정 ---
# 중요: 사용자 환경에 맞게 경로를 수정해주세요.
sub = '01' # 피험자 번호
project_dir = os.path.abspath('.') # 현재 프로젝트 디렉토리
output_dir = os.path.join(project_dir, f'output/pilot/sub-{sub}') # fMRIPrep 결과물이 있는 상위 폴더
fmriprep_dir = os.path.join(output_dir, 'func') # 전처리된 기능 영상 폴더
analysis_dir = os.path.join(project_dir, f'derivatives/sub-{sub}') # 분석 결과를 저장할 폴더

os.makedirs(analysis_dir, exist_ok=True)

# --- 분석 파라미터 ---
n_runs = 6 # 총 run의 수
TR = 1.5 # TR (초)
vols_to_drop = 4  # 각 run의 시작에서 제거할 볼륨 수 (자기장 안정화 시간)
n_colors = 8 # 자극 색상의 수

# --- 파일 경로 리스트 생성 ---
# 기능 영상, 이벤트 파일, confound 파일의 전체 경로를 생성합니다.
func_imgs_paths = [os.path.join(fmriprep_dir, f'sub-{sub}_task-rsvp_run-{i+1}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz') for i in range(n_runs)]
event_files_paths = [os.path.join(project_dir, f'pilot/sub-{sub}/func/sub-{sub}_task-rsvp_run-{i+1}_events.tsv') for i in range(n_runs)]
confound_files_paths = [os.path.join(fmriprep_dir, f'sub-{sub}_task-rsvp_run-{i+1}_desc-confounds_timeseries.tsv') for i in range(n_runs)]
anat_mask_path = os.path.join(fmriprep_dir, f'sub-{sub}_task-rsvp_run-1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz')

# --- ROI 경로 --- 
# 중요: 실제 아틀라스 기반 ROI 파일 경로로 교체해야 합니다.
roi_files = {
    'V1': os.path.join(analysis_dir, 'roi/V1_mask.nii.gz'),
    'V2': os.path.join(analysis_dir, 'roi/V2_mask.nii.gz'),
    'V3': os.path.join(analysis_dir, 'roi/V3_mask.nii.gz'),
    'hV4': os.path.join(analysis_dir, 'roi/hV4_mask.nii.gz')
}
# 만약 ROI 파일이 없다면, 데모를 위해 빈 더미 파일을 생성합니다.
os.makedirs(os.path.join(analysis_dir, 'roi'), exist_ok=True)
for roi_path in roi_files.values():
    if not os.path.exists(roi_path):
        dummy_roi = nib.Nifti1Image(np.zeros(nib.load(anat_mask_path).shape), nib.load(anat_mask_path).affine)
        dummy_roi.to_filename(roi_path)
        print("madeROI")

In [2]:
def prepare_glm_inputs(func_paths, event_paths, confound_paths, vols_to_drop):
    """GLM 분석을 위한 입력 데이터(기능 영상, 이벤트, confound)를 준비합니다.
    B&H (2009) 지침에 따라 각 run의 초기 볼륨들을 제거하고, 그에 맞게 이벤트 시간을 보정합니다."""
    prepared_funcs, prepared_events, prepared_confounds = [], [], []

    # fMRIPrep confound 파일에서 사용할 변수 목록 (움직임 파라미터 + aCompCor)
    confound_cols = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 
                     'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03', 'a_comp_cor_04']

    for i, (func_p, event_p, confound_p) in enumerate(zip(func_paths, event_paths, confound_paths)):
        # 1. 기능 영상 처리: 초기 볼륨(vols_to_drop) 제거
        func_img = nib.load(func_p)
        func_img_trimmed = index_img(func_img, slice(vols_to_drop, None))
        prepared_funcs.append(func_img_trimmed)

        # 2. 이벤트 파일 처리: 제거된 볼륨만큼 onset 시간 보정
        events_df = pd.read_csv(event_p, sep='	')
        events_df['onset'] -= vols_to_drop * TR
        events_df = events_df[events_df['trial_type'].str.contains('color|blank')] # 분석에 필요한 trial만 선택
        prepared_events.append(events_df)

        # 3. Confound 파일 처리: 초기 볼륨에 해당하는 행 제거 및 주요 변수 선택
        confounds_df = pd.read_csv(confound_p, sep='	')
        confounds_df_trimmed = confounds_df.iloc[vols_to_drop:].reset_index(drop=True)
        selected_confounds = confounds_df_trimmed[confound_cols].fillna(0) # NaN 값을 0으로 채움
        prepared_confounds.append(selected_confounds)

    return prepared_funcs, prepared_events, prepared_confounds

func_imgs, events, confounds = prepare_glm_inputs(func_imgs_paths, event_files_paths, confound_files_paths, vols_to_drop)

# FIR 모델 설정: B&H(2009)는 자극 제시 후 긴 시간(12초) 동안의 반응을 모델링했습니다.
# TR이 1.5초이므로 8개의 lag를 사용하여 1.5초 ~ 12초 구간을 커버합니다.
fir_delays = np.arange(1, 9) 

fmri_glm = FirstLevelModel(
    t_r=TR,
    hrf_model='fir', # HRF 형태를 가정하지 않는 FIR 모델 사용
    fir_delays=fir_delays, # 8개의 lag (1.5s, 3.0s, ..., 12.0s)
    drift_model='cosine', # 저주파 노이즈 제거
    high_pass=0.01,
    mask_img=anat_mask_path, # 뇌 영역 내에서만 분석 수행
    noise_model='ar1', # 시간적 자기상관 보정
    standardize=False, # B&H(2009)는 표준화를 수행하지 않음
    n_jobs=-1 # 가능한 모든 CPU 코어 사용
)

fmri_glm = fmri_glm.fit(run_imgs=func_imgs, events=events, confounds=confounds)

/tmp/ipykernel_2539831/1918126500.py:48: UserWarning: Mean values of 0 observed. The data have probably been centered. Scaling might not work as expected.
  fmri_glm = fmri_glm.fit(run_imgs=func_imgs, events=events, confounds=confounds)


In [ ]:
# --- 베타 맵 계산 및 저장 ---
# 인코딩 모델의 입력은 z-점수가 아닌 각 조건의 반응 크기를 나타내는 '베타(β) 계수'입니다.
beta_map_dir = os.path.join(analysis_dir, 'betas')
os.makedirs(beta_map_dir, exist_ok=True)

design_matrices = fmri_glm.design_matrices_

# 각 run의 regressor 수를 계산하여 전체 design matrix에서의 위치를 찾을 준비를 합니다.
n_regressors_per_run = [dm.shape[1] for dm in design_matrices]
total_regressors = sum(n_regressors_per_run)

for run_idx in range(n_runs):
    # 현재 run의 regressor들이 전체 design matrix에서 시작되는 지점(offset)을 계산합니다.
    run_offset = sum(n_regressors_per_run[:run_idx])
    
    for color_idx in range(1, n_colors + 1):
        for lag_idx, lag in enumerate(fir_delays):
            condition = f'color_{color_idx}'
            contrast_col = f'{condition}_delay_{lag_idx+1}'
            
            # 특정 run의 design matrix에 해당 조건이 있는지 확인합니다.
            if contrast_col in design_matrices[run_idx].columns:
                # 전체 design matrix에 대한 contrast vector를 0으로 초기화합니다.
                contrast_vector = np.zeros(total_regressors)
                
                # 현재 run 내에서 원하는 regressor의 위치(index)를 찾습니다.
                col_idx_in_run = design_matrices[run_idx].columns.get_loc(contrast_col)
                
                # 전체 design matrix에서의 최종 위치를 계산하여 해당 위치의 값을 1로 설정합니다.
                final_col_idx = run_offset + col_idx_in_run
                contrast_vector[final_col_idx] = 1
                
                # 'effect_size'는 베타 계수를 직접 반환합니다.
                # 생성된 contrast vector를 사용하고, 에러를 유발한 run_idx 인자는 제거합니다.
                beta_map = fmri_glm.compute_contrast(contrast_vector, 
                                                     output_type='effect_size')
                
                out_fname = f'sub-{sub}_run-{run_idx+1}_color-{color_idx}_lag-{lag_idx+1}_beta.nii.gz'
                beta_map.to_filename(os.path.join(beta_map_dir, out_fname))

## 3. ROI 데이터 추출
인코딩 모델에 사용하기 위해, 각 색상 조건에 대한 단일 반응 값을 얻습니다.
이를 위해 FIR 모델에서 얻은 여러 lag의 베타 값들을 평균냅니다.

In [ ]:
# --- Lag에 대한 베타 평균내기 ---
# 각 색상 조건에 대해 8개 lag의 베타를 평균하여 하나의 안정적인 반응 추정치를 얻습니다.
avg_beta_dir = os.path.join(analysis_dir, 'betas_avg')
os.makedirs(avg_beta_dir, exist_ok=True)

for run_idx in range(n_runs):
    for color_idx in range(1, n_colors + 1):
        beta_maps_for_color = []
        for lag_idx in range(len(fir_delays)):
            beta_map_path = os.path.join(avg_beta_dir, f'sub-{sub}_run-{run_idx+1}_color-{color_idx}_lag-{lag_idx+1}_beta.nii.gz')
            if os.path.exists(beta_map_path):
                beta_maps_for_color.append(nib.load(beta_map_path))
        
        # 해당 색상에 대한 베타 맵이 하나라도 존재하면 평균 이미지 생성
        if beta_maps_for_color:
            avg_beta_map = nib.Nifti1Image(
                np.mean(np.stack([b.get_fdata() for b in beta_maps_for_color], axis=-1), axis=-1),
                affine=beta_maps_for_color[0].affine
            )
            out_fname = f'sub-{sub}_run-{run_idx+1}_color-{color_idx}_avg_beta.nii.gz'
            avg_beta_map.to_filename(os.path.join(avg_beta_dir, out_fname))



## 4. Forward Encoding Model (B&H 2009)
분석의 핵심 단계입니다. Voxel 반응(B)과 이론적인 채널 출력(C)의 관계를 모델링하는 가중치(W)를 학습합니다.
- **채널(C) 정의**: 6개의 이상적인 색상 채널을 정의합니다.
- **학습**: `W_hat = B1 * C1' * (C1 * C1')^-1`
- **테스트**: `C_hat_2 = (W_hat' * W_hat)^-1 * W_hat' * B2`
- **디코딩**: 재구성된 채널 반응(`C_hat_2`)과 실제 채널 반응(`C`) 사이의 상관관계가 가장 높은 색상을 선택합니다.

In [ ]:
def create_color_channels(n_channels=6, n_colors=8):
    """B&H (2009)에 기술된 6개의 색상 채널을 생성합니다.
    각 채널은 색상 원(hue circle)에서 특정 색상을 선호하며, 반파 정류 및 제곱된 사인 함수 형태를 가집니다."""
    channel_centers = np.linspace(0, 360, n_channels, endpoint=False)
    stim_hues = np.linspace(0, 360, n_colors, endpoint=False) # 8개 자극 색상의 각도
    
    # C: 채널 행렬, shape=(k_channels, n_colors)
    C = np.zeros((n_channels, len(stim_hues)))
    for i, center in enumerate(channel_centers):
        # 각 채널의 중심(선호 색상)으로부터의 거리(각도 차이)에 따라 사인 함수 계산
        channel_response = np.sin(np.deg2rad(stim_hues - center))
        # 반파 정류(음수 값을 0으로) 및 제곱. 이를 통해 채널의 선택성을 높임.
        channel_response[channel_response < 0] = 0
        C[i, :] = channel_response ** 2
    return C

# C: 이론적인 채널 반응 행렬 (k x n_colors)
C = create_color_channels()

# 생성된 채널 시각화
plt.figure(figsize=(8, 5))
for i in range(C.shape[0]):
    plt.plot(np.linspace(0, 315, 8), C[i, :], 'o-', label=f'Channel {i+1}')
plt.title('Idealized Color Channel Responses (B&H 2009)')
plt.xlabel('Stimulus Hue (degrees)')
plt.ylabel('Channel Output')
plt.legend()
plt.show()

In [ ]:
def run_decoding_analysis(roi_name, roi_path, avg_beta_dir):
    """지정된 ROI에 대해 leave-one-run-out 교차 검증 디코딩을 수행합니다."""
    masker = NiftiMasker(mask_img=roi_path, standardize=False) # ROI 내 voxel 데이터 추출
    
    # 1. ROI 내 모든 베타 값 로드
    # all_betas shape: (n_runs, n_colors, n_voxels)
    all_betas = []
    for run_idx in range(n_runs):
        run_betas_paths = [os.path.join(avg_beta_dir, f'sub-{sub}_run-{run_idx+1}_color-{c_idx}_avg_beta.nii.gz') for c_idx in range(1, n_colors + 1)]
        run_betas_masked = masker.fit_transform(run_betas_paths) # shape: (n_colors, n_voxels)
        all_betas.append(run_betas_masked)
    all_betas = np.array(all_betas)
    n_voxels = all_betas.shape[2]
    print(f'[{roi_name}] ROI 로드 완료. Voxel 수: {n_voxels}')
    
    accuracies = []
    true_labels = np.arange(n_colors)
    
    # 2. Leave-one-run-out 교차 검증 시작
    for test_run_idx in range(n_runs):
        train_run_indices = [i for i in range(n_runs) if i != test_run_idx]
        
        # 훈련 데이터 준비
        # B1: 훈련 데이터의 voxel 반응 행렬 (m_voxels, n_samples)
        B1 = all_betas[train_run_indices].reshape(-1, n_voxels).T
        # C1: 훈련 데이터의 채널 출력 행렬 (k_channels, n_samples)
        C1 = np.tile(C, len(train_run_indices))
        
        # 테스트 데이터 준비
        # B2: 테스트 데이터의 voxel 반응 행렬 (m_voxels, n_colors)
        B2 = all_betas[test_run_idx].T
        
        # 3. 모델 학습: OLS를 사용하여 가중치(W_hat) 추정
        W_hat = B1 @ C1.T @ np.linalg.pinv(C1 @ C1.T) # shape: (m, k)
        
        # 4. 모델 테스트: 학습된 가중치를 사용하여 테스트 데이터의 채널 반응(C_hat_2) 재구성
        C_hat_2 = np.linalg.pinv(W_hat.T @ W_hat) @ W_hat.T @ B2 # shape: (k, n_colors)
        
        # 5. 디코딩: 재구성된 채널 반응과 실제 채널 프로파일 간의 상관관계 계산
        correlation_matrix = np.corrcoef(C_hat_2.T, C.T)[n_colors:, :n_colors]
        predicted_labels = np.argmax(correlation_matrix, axis=1)
        
        # 6. 정확도 계산
        accuracy = np.mean(predicted_labels == true_labels)
        accuracies.append(accuracy)
        
    mean_accuracy = np.mean(accuracies)
    print(f'[{roi_name}] LOO 평균 정확도: {mean_accuracy:.3f}')
    return mean_accuracy, all_betas

# 모든 ROI에 대해 분석 실행
results = {}
for roi_name, roi_path in roi_files.items():
    if os.path.exists(roi_path) and nib.load(roi_path).get_fdata().sum() > 0:
        mean_acc, roi_betas = run_decoding_analysis(roi_name, roi_path, avg_beta_dir)
        results[roi_name] = {'accuracy': mean_acc, 'betas': roi_betas}
    else:
        print(f'[{roi_name}] ROI 파일이 비어있거나 존재하지 않아 스킵합니다.')

## 5. Significance Testing (Permutation Test)
관찰된 디코딩 정확도가 우연에 의한 것인지 확인하기 위해 순열 검증을 수행합니다.
훈련 데이터의 색상 레이블을 무작위로 섞어(shuffling) 우연 수준의 정확도 분포(null distribution)를 만듭니다.

In [ ]:
def run_permutation_test(roi_name, all_betas, n_permutations=1000):
    """디코딩 분석에 대한 순열 검증을 수행하여 p-value를 계산합니다."""
    n_runs, n_colors, n_voxels = all_betas.shape
    null_accuracies = []
    true_labels = np.arange(n_colors)
    print(f'[{roi_name}] 순열 검증 시작 (n={n_permutations})...')
    
    for perm in range(n_permutations):
        perm_accuracies = []
        # 교차 검증 루프 내에서 레이블을 섞음
        for test_run_idx in range(n_runs):
            train_run_indices = [i for i in range(n_runs) if i != test_run_idx]
            
            # 중요: 훈련 단계에서만 채널-색상 매핑을 무작위로 섞습니다.
            permuted_C = C[:, np.random.permutation(n_colors)]
            
            B1 = all_betas[train_run_indices].reshape(-1, n_voxels).T
            C1_perm = np.tile(permuted_C, len(train_run_indices))
            B2 = all_betas[test_run_idx].T
            
            # 순열된 레이블로 모델 학습 및 테스트
            W_hat = B1 @ C1_perm.T @ np.linalg.pinv(C1_perm @ C1_perm.T)
            C_hat_2 = np.linalg.pinv(W_hat.T @ W_hat) @ W_hat.T @ B2
            
            # 디코딩은 원래 채널(C)을 기준으로 수행
            correlation_matrix = np.corrcoef(C_hat_2.T, C.T)[n_colors:, :n_colors]
            predicted_labels = np.argmax(correlation_matrix, axis=1)
            perm_accuracies.append(np.mean(predicted_labels == true_labels))
            
        null_accuracies.append(np.mean(perm_accuracies))
        
    # p-value 계산: 실제 정확도보다 높거나 같은 null 정확도의 비율
    observed_accuracy = results[roi_name]['accuracy']
    p_value = (np.sum(null_accuracies >= observed_accuracy) + 1) / (n_permutations + 1)
    print(f'[{roi_name}] 순열 검증 완료: p-value = {p_value:.4f}')
    results[roi_name]['p_value'] = p_value
    results[roi_name]['null_dist'] = null_accuracies
    return p_value, null_accuracies

# 분석 결과가 있는 ROI에 대해서만 순열 검증 실행
for roi_name in results:
    run_permutation_test(roi_name, results[roi_name]['betas'])

## 6. PCA Visualization
B&H (2009)의 보조 분석(complementary analysis)으로, voxel 반응 패턴에 PCA를 적용합니다.
PC1과 PC2로 만들어진 공간에 8개 색상에 대한 반응을 플롯했을 때, 색상환과 유사한 원형 구조가 나타나는지 확인하여 인코딩 모델의 결과를 시각적으로 검증합니다.

In [ ]:
def plot_pca_results(roi_name, all_betas):
    """Voxel 반응 패턴에 PCA를 적용하고 PC1-PC2 공간에 시각화합니다."""
    # 모든 run에 걸쳐 베타를 평균하여 각 색상에 대한 평균 반응 패턴을 얻습니다.
    mean_betas_per_color = np.mean(all_betas, axis=0) # shape: (n_colors, n_voxels)
    
    pca = PCA(n_components=2)
    pc_coords = pca.fit_transform(mean_betas_per_color)
    
    plt.figure(figsize=(6, 6))
    # 8개 색상을 hsv 컬러맵을 사용하여 시각화
    hues = np.linspace(0, 1, 8, endpoint=False)
    colors = [plt.cm.hsv(h) for h in hues]
    
    plt.scatter(pc_coords[:, 0], pc_coords[:, 1], c=colors, s=100, edgecolor='k')
    for i, (x, y) in enumerate(pc_coords):
        plt.text(x*1.15, y*1.15, f'C{i+1}', fontsize=12, ha='center', va='center')
        
    plt.title(f'PCA of Voxel Responses in {roi_name}')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True)
    plt.show()

# 분석 결과가 있는 ROI에 대해서만 PCA 시각화 실행
for roi_name in results:
    plot_pca_results(roi_name, results[roi_name]['betas'])